In [2]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

# from tensorboardX import SummaryWriter      

import torchvision
import torchvision.transforms as transforms

import sys

sys.path.append(os.path.abspath(".."))

from models import *

import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   
os.environ["CUDA_VISIBLE_DEVICES"]="0"

global best_prec
use_gpu = torch.cuda.is_available()
print('=> Building model...')
    
    
batch_size = 128
model_name = "VGG16_quant"
model = VGG16_quant()
print(model)

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)

        input, target = input.cuda(), target.cuda()

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.cuda(), target.cuda()

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].view(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

=> Building model...
VGG_quant(
  (features): Sequential(
    (0): QuantConv2d(
      3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): QuantConv2d(
      64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): QuantConv2d(
      64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
      (weight_quant): weight_quantize_fn()
    )
    (8): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (9): ReLU(inplace=True)
    (10): QuantConv2d(
      128, 128, kernel_size=(3, 3), stride

In [3]:
model = nn.DataParallel(model).cuda()
all_params = checkpoint['state_dict']
model.load_state_dict(all_params, strict=False)
criterion = nn.CrossEntropyLoss().cuda()
validate(testloader, model, criterion)

NameError: name 'checkpoint' is not defined

In [5]:
PATH = "../saved_models/checkpoint.pth"
checkpoint = torch.load(PATH)
model = model.module

model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

C:\tmp\ipykernel_31148\2624006234.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH)



Test set: Accuracy: 8149/10000 (81%)



In [6]:
#### Prune all the QuantConv2D layers' 90% weights with 1) unstructured, and 2) structured manner.
import torch.nn.utils.prune as prune

for i, feature in enumerate(model.features):
  if isinstance(feature, QuantConv2d):
    prune.l1_unstructured(feature, name="weight", amount=0.8)


In [7]:
print(list(model.features[40].named_parameters())) # check whether there is mask, weight_org, ...
print(model.features[40].weight) # check whether there are many zeros

[('act_alpha', Parameter containing:
tensor(7.8643, device='cuda:0', requires_grad=True)), ('weight_q', Parameter containing:
tensor([[[[-0.4213, -1.2640,  0.4213],
          [ 0.4213,  1.2640, -1.6853],
          [-0.4213, -0.8427,  0.8427]],

         [[-0.4213,  0.8427, -0.0000],
          [-1.2640,  0.8427,  0.4213],
          [-0.4213,  2.1066,  1.2640]],

         [[ 1.2640,  1.2640, -0.8427],
          [-1.2640, -1.2640, -1.2640],
          [ 0.8427,  1.2640,  0.4213]],

         ...,

         [[ 0.4213,  0.4213,  1.6853],
          [ 0.4213,  2.1066, -0.4213],
          [ 0.8427,  1.2640, -1.2640]],

         [[-0.4213, -1.2640,  0.8427],
          [ 1.2640, -0.8427, -0.4213],
          [-0.0000,  0.8427,  0.8427]],

         [[ 0.8427,  0.4213, -1.6853],
          [-0.8427, -1.2640,  0.0000],
          [ 0.8427, -2.1066, -0.4213]]],


        [[[ 0.4213, -0.8427, -1.6853],
          [-0.4213,  0.0000, -1.2640],
          [-1.2640, -0.8427, -1.2640]],

         [[ 0.8427, -0.4

In [8]:
### Check sparsity ###
mask1 = model.features[40].weight_mask
sparsity_mask1 = (mask1 == 0).sum() / mask1.nelement()

print("Sparsity level: ", sparsity_mask1)

Sparsity level:  tensor(0.8000, device='cuda:0')


In [10]:
# Get sparsity per conv layer
cnt = 1
for feature in model.features:
    if isinstance(feature, QuantConv2d):
        mask = feature.weight_mask
        sparsity_mask = (mask == 0).sum() / mask.nelement()
        print(f"Conv layer: {cnt} \t Sparsity: {sparsity_mask}")
        cnt += 1

Conv layer: 1 	 Sparsity: 0.7997685074806213
Conv layer: 2 	 Sparsity: 0.7999945878982544
Conv layer: 3 	 Sparsity: 0.7999945878982544
Conv layer: 4 	 Sparsity: 0.8000013828277588
Conv layer: 5 	 Sparsity: 0.8000013828277588
Conv layer: 6 	 Sparsity: 0.7999996542930603
Conv layer: 7 	 Sparsity: 0.7999996542930603
Conv layer: 8 	 Sparsity: 0.7999996542930603
Conv layer: 9 	 Sparsity: 0.8000000715255737
Conv layer: 10 	 Sparsity: 0.8000000715255737
Conv layer: 11 	 Sparsity: 0.8000000715255737
Conv layer: 12 	 Sparsity: 0.8000000715255737
Conv layer: 13 	 Sparsity: 0.8000000715255737


In [9]:
## check accuracy after pruning
model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))


Test set: Accuracy: 999/10000 (10%)



In [11]:
## Start finetuning (training here), and see how much you can recover your accuracy ##
## You can change hyper parameters such as epochs or lr #

lr = 4e-2
weight_decay = 1e-4
epochs = 20
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 8.768 (8.768)	Data 7.998 (7.998)	Loss 3.7013 (3.7013)	Prec 26.562% (26.562%)
Epoch: [0][100/391]	Time 0.094 (0.183)	Data 0.001 (0.080)	Loss 0.9100 (1.4775)	Prec 64.844% (50.480%)
Epoch: [0][200/391]	Time 0.087 (0.140)	Data 0.000 (0.041)	Loss 0.6918 (1.1295)	Prec 76.562% (61.890%)
Epoch: [0][300/391]	Time 0.093 (0.126)	Data 0.000 (0.028)	Loss 0.7109 (0.9855)	Prec 74.219% (66.546%)
Validation starts
Test: [0/79]	Time 6.325 (6.325)	Loss 0.5533 (0.5533)	Prec 82.812% (82.812%)
 * Prec 77.080% 
best acc: 77.080000
Epoch: [1][0/391]	Time 7.671 (7.671)	Data 7.268 (7.268)	Loss 0.7216 (0.7216)	Prec 76.562% (76.562%)
Epoch: [1][100/391]	Time 0.096 (0.179)	Data 0.001 (0.073)	Loss 0.4820 (0.5739)	Prec 82.812% (80.252%)
Epoch: [1][200/391]	Time 0.096 (0.139)	Data 0.001 (0.037)	Loss 0.6308 (0.5721)	Prec 75.781% (80.239%)
Epoch: [1][300/391]	Time 0.093 (0.125)	Data 0.001 (0.025)	Loss 0.7226 (0.5635)	Prec 71.094% (80.575%)
Validation starts
Test: [0/79]	Time 6.531 (6.531)	Loss 0.

In [13]:
## check your accuracy again after finetuning

# Load best model
PATH = "./result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)

model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

C:\tmp\ipykernel_31148\809325536.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH)



Test set: Accuracy: 8673/10000 (87%)



In [14]:
## Send an image and use prehook to grab the inputs of all the QuantConv2d layers

class SaveOutput:
    def __init__(self):
        self.outputs = []
    def __call__(self, module, module_in):
        self.outputs.append(module_in)
    def clear(self):
        self.outputs = []  
        
######### Save inputs from selected layer ##########
save_output = SaveOutput()
i = 0

for layer in model.modules():
    i = i+1
    if isinstance(layer, QuantConv2d):
        print(i,"-th layer prehooked")
        layer.register_forward_pre_hook(save_output)             
####################################################

dataiter = iter(testloader)
images, labels = dataiter.next()
images = images.to(device)
out = model(images)

3 -th layer prehooked
7 -th layer prehooked
12 -th layer prehooked
16 -th layer prehooked
21 -th layer prehooked
25 -th layer prehooked
29 -th layer prehooked
34 -th layer prehooked
38 -th layer prehooked
42 -th layer prehooked
47 -th layer prehooked
51 -th layer prehooked
55 -th layer prehooked


AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute 'next'

In [15]:
##### Find "weight_int" for features[3] ####
w_bit = 4
weight_q = model.features[3].weight_q
w_alpha = model.features[3].weight_quant.wgt_alpha
w_delta = w_alpha /(2**(w_bit-1)-1)

weight_int = weight_q / w_delta
print(weight_int)

tensor([[[[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000, -1.0000,  0.0000]],

         [[ 0.0000,  6.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [-7.0000,  0.0000,  0.0000]],

         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [-5.0000,  0.0000, -2.0000]],

         ...,

         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000]],

         [[-2.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  5.0000],
          [ 0.0000,  6.0000,  0.0000]],

         [[ 0.0000,  0.0000,  5.0000],
          [ 0.0000,  0.0000,  4.0000],
          [ 0.0000,  0.0000, -0.0000]]],


        [[[ 0.0000,  7.0000,  7.0000],
          [ 0.0000,  7.0000,  7.0000],
          [ 7.0000,  7.0000,  0.0000]],

         [[ 0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000],
          [ 7.0000,  0.0000,  0.0000]],

         [[ 0.0000, -4.0000,  0

In [ ]:
#### check your sparsity for weight_int is near 80% #####
#### Your sparsity could be >90% after quantization #####
sparsity_weight_int = (weight_int == 0).sum() / weight_int.nelement()
print("Sparsity level: ", sparsity_weight_int)

Sparsity level:  tensor(0.8143, device='cuda:0')


In [21]:
# Try again but structured pruning this time
# Load the original unpruned model
PATH = "../saved_models/checkpoint.pth"
checkpoint = torch.load(PATH)

model = VGG16_quant()

model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

C:\tmp\ipykernel_31148\2150592374.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH)



Test set: Accuracy: 8149/10000 (81%)



In [24]:
# Apply structured pruning
for i, feature in enumerate(model.features):
  if isinstance(feature, QuantConv2d):
    prune.ln_structured(feature, name="weight", n=1, amount=0.8, dim=0)

In [25]:
# Retrain
## Start finetuning (training here), and see how much you can recover your accuracy ##
## You can change hyper parameters such as epochs or lr #

lr = 4e-2
weight_decay = 1e-4
epochs = 20
best_prec = 0

#model = nn.DataParallel(model).cuda()
model.cuda()
criterion = nn.CrossEntropyLoss().cuda()
optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
#cudnn.benchmark = True

if not os.path.exists('result'):
    os.makedirs('result')
fdir = 'result/'+str(model_name)
if not os.path.exists(fdir):
    os.makedirs(fdir)
        

for epoch in range(0, epochs):
    adjust_learning_rate(optimizer, epoch)

    train(trainloader, model, criterion, optimizer, epoch)
    
    # evaluate on test set
    print("Validation starts")
    prec = validate(testloader, model, criterion)

    # remember best precision and save checkpoint
    is_best = prec > best_prec
    best_prec = max(prec,best_prec)
    print('best acc: {:1f}'.format(best_prec))
    save_checkpoint({
        'epoch': epoch + 1,
        'state_dict': model.state_dict(),
        'best_prec': best_prec,
        'optimizer': optimizer.state_dict(),
    }, is_best, fdir)

Epoch: [0][0/391]	Time 7.135 (7.135)	Data 6.711 (6.711)	Loss 2.5654 (2.5654)	Prec 12.500% (12.500%)
Epoch: [0][100/391]	Time 0.095 (0.167)	Data 0.001 (0.067)	Loss 1.6931 (1.9487)	Prec 39.844% (24.288%)
Epoch: [0][200/391]	Time 0.097 (0.132)	Data 0.001 (0.034)	Loss 1.5888 (1.8131)	Prec 34.375% (29.213%)
Epoch: [0][300/391]	Time 0.108 (0.121)	Data 0.001 (0.023)	Loss 1.4573 (1.7303)	Prec 43.750% (32.418%)
Validation starts
Test: [0/79]	Time 6.322 (6.322)	Loss 1.4245 (1.4245)	Prec 47.656% (47.656%)
 * Prec 45.370% 
best acc: 45.370000
Epoch: [1][0/391]	Time 7.271 (7.271)	Data 6.883 (6.883)	Loss 1.5130 (1.5130)	Prec 39.844% (39.844%)
Epoch: [1][100/391]	Time 0.097 (0.168)	Data 0.001 (0.069)	Loss 1.2078 (1.4465)	Prec 57.812% (44.918%)
Epoch: [1][200/391]	Time 0.102 (0.133)	Data 0.001 (0.035)	Loss 1.3148 (1.4157)	Prec 53.906% (46.716%)
Epoch: [1][300/391]	Time 0.096 (0.121)	Data 0.002 (0.024)	Loss 1.1551 (1.3816)	Prec 54.688% (48.092%)
Validation starts
Test: [0/79]	Time 6.240 (6.240)	Loss 1.

In [26]:
## check your accuracy again after finetuning

# Load best model
PATH = "./result/VGG16_quant/model_best.pth.tar"
checkpoint = torch.load(PATH)

model.load_state_dict(checkpoint['state_dict'])
device = torch.device("cuda") 

model.cuda()
model.eval()

test_loss = 0
correct = 0

with torch.no_grad():
    for data, target in testloader:
        data, target = data.to(device), target.to(device) # loading to GPU
        output = model(data)
        pred = output.argmax(dim=1, keepdim=True)  
        correct += pred.eq(target.view_as(pred)).sum().item()

test_loss /= len(testloader.dataset)

print('\nTest set: Accuracy: {}/{} ({:.0f}%)\n'.format(
        correct, len(testloader.dataset),
        100. * correct / len(testloader.dataset)))

C:\tmp\ipykernel_31148\809325536.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(PATH)



Test set: Accuracy: 6985/10000 (70%)

